# CA noise robustness: forward return variant

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Defining the utils**

**COMMENT: restart after having run this cell, so it is in memory**

**Description**

1. Change the results path for the number hidden layer in each model, iteratively : results_CA_0, results_CA_1, results_CA_2, results_CA_0
2. Run the code that defines the desired model with the number of hidden layers
3. Run the codes that preprocess the data for the models.
4. Run the code for the cross-validation, which makes prediction on the validation set. The performance evaluation metrcs are saved in the results path.Remember to change the results path for the cross-validation.
5. Run the the code that predicts the desired model with the desired number of hidden layers. remember to change the results path for the out-of-sample  predictions on the test set. This code makes predictions for each latent latent factor across a time-series split, and the predicted results are saved.

**sliding window with validation**

In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx, val_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 val_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              val_start_idx, val_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            val_idx = dates[(dates[self.date_idx] > days[val_start])
                              & (dates[self.date_idx] <= days[val_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),val_idx.to_numpy(), test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits



class DateTimeSeriesCV:
    """Generates tuples of train_idx,val_idx and test_idx pairs
    Assumes the index is of datetime type
    purges overlapping outcomes"""

    def __init__(self, n_splits=3, train_period_length=126, val_period_length=21, test_period_length=21, lookahead=None, shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle

    def split(self, X, y=None, groups=None):
        unique_dates = X.index.unique()
        days = sorted(unique_dates, reverse=True)

        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead

            if train_start_idx >= len(days):
                break  # Stop if we've gone beyond the dataset

            split_idx.append([train_start_idx, train_end_idx, val_start_idx, val_end_idx, test_start_idx, test_end_idx])

        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:
            train_idx = X.index[(X.index > days[train_start]) & (X.index <= days[train_end])]
            val_idx = X.index[(X.index > days[val_start]) & (X.index <= days[val_end])]
            test_idx = X.index[(X.index > days[test_start]) & (X.index <= days[test_end])]

            if self.shuffle:
                train_idx = np.random.permutation(train_idx)

            yield train_idx, val_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


In [ ]:
!pip install tables

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
from time import time
from pathlib import Path
from itertools import product
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import time
import random
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Reshape, BatchNormalization, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import quantile_transform
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

from utils import format_time, MultipleTimeSeriesCV
from utils import format_time, DateTimeSeriesCV

from numpy.linalg import inv
from google.colab import drive




**Setting up TPU**

In [ ]:
# # Set up the TPU
# try:
#     tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
#     print('Running on TPU ', tpu.master())

#     # Always connect to the TPU cluster and initialize the TPU system
#     tf.config.experimental_connect_to_cluster(tpu)
#     tf.tpu.experimental.initialize_tpu_system(tpu)

#     strategy = tf.distribute.TPUStrategy(tpu)
#     print("All TPU devices: ", tf.config.experimental.list_logical_devices('TPU'))
# except ValueError:
#     raise BaseException('ERROR: Not connected to a TPU runtime; select TPU from the "Runtime > Change runtime type" menu.')



**defining my results path**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/Robustness_check_predictive')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)


In [ ]:

# ###########################
# ## MOUNT TO GOOGLE DRIVE ##
# ###########################

# # Mount Google Drive
# drive.mount('/content/drive')

# ################################
# ## LOAD DATA FOR BETA NETWORK ##
# ################################


# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'

# # Make sure merged_df1 is a DataFrame and not a function or any other type.
# merged_df_padded = pd.read_parquet(file_path)

# # Assuming merged_df_padded is already defined
# # Ensure the date level of the MultiIndex is of datetime type
# merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)

# # Filter the DataFrame to include only rows from 1979 and onwards
# merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1960-01-01']

# print(merged_df_padded)
# print(merged_df_padded.index)
# print(merged_df_padded.shape)
# print(type(merged_df_padded))

# #
# #################################
# ## LOAD DATA FOR FACTOR NETWORK##
# #################################

# # Define the file path
# file_path = '/content/drive/My Drive/Tidy Finance ML dataset/x_t_preprocessed.parquet'

# # Load the Parquet file into a DataFrame
# portfolio_rets_df = pd.read_parquet(file_path)



# # Assuming portfolio_rets_df is already defined and loaded
# # Ensure the date index is of datetime type
# portfolio_rets_df.index = pd.to_datetime(portfolio_rets_df.index)

# # Filter the DataFrame to include only rows from 1979 and onwards
# portfolio_rets_df = portfolio_rets_df.loc[portfolio_rets_df.index >= '1960-01-01']

# print(portfolio_rets_df)
# print(portfolio_rets_df.index)
# print(portfolio_rets_df.shape)
# print(type(portfolio_rets_df))




In [ ]:
# import pandas as pd
# import numpy as np

# # Mount Google Drive
# drive.mount('/content/drive')

# # Load data for beta network
# beta_file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta.parquet'
# merged_df_padded = pd.read_parquet(beta_file_path)
# merged_df_padded.index = merged_df_padded.index.set_levels(pd.to_datetime(merged_df_padded.index.levels[0]), level=0)
# merged_df_padded = merged_df_padded.loc[merged_df_padded.index.get_level_values('date') >= '1975-01-01']

# # Remove rows with any -2 values
# merged_df_cleaned = merged_df_padded[~(merged_df_padded == -2).any(axis=1)]




# print(merged_df_cleaned)
# print(merged_df_cleaned.index)
# print(merged_df_cleaned.shape)
# print(type(merged_df_cleaned))


In [ ]:
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

# # Define the list of characteristic codes
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Assuming merged_df_padded is already defined and loaded

# # Replace -2 with NaN to correctly handle missing values
# merged_df_padded.replace(-2, np.nan, inplace=True)

# # Parameters for noise generation
# unique_permnos = merged_df_padded.index.get_level_values('permno').unique()
# unique_dates = merged_df_padded.index.get_level_values('date').unique()
# N = len(unique_permnos)  # Number of unique permnos
# T = len(unique_dates)  # Number of time periods
# Pc = 5  # Number of noise characteristics

# # Calculate the standard deviation of the specified columns ignoring NaNs
# characteristics_std = merged_df_padded[characteristics].std(skipna=True).mean()

# # Simulate noise characteristics
# rho = np.random.uniform(0.9, 1, Pc)
# c = np.zeros((N, Pc, T))

# # Generate the noise characteristics
# for t in tqdm(range(1, T), desc="Generating noise characteristics"):
#     c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

# # Normalize noise characteristics
# c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

# # Scale noise characteristics
# c_scaled = c_normalized * characteristics_std

# # Check the shape of c_scaled
# print(c_scaled.shape)  # (N, Pc, T)

# # Reshape scaled characteristics to fit the DataFrame structure
# c_scaled = c_scaled.transpose(2, 0, 1)  # Transpose to (T, N, Pc)
# c_scaled = c_scaled.reshape(T * N, Pc)  # Flatten to (T*N, Pc)

# # Create a new MultiIndex to match the merged_df_padded index
# dates = np.repeat(unique_dates, N)
# permnos = np.tile(unique_permnos, T)

# # Verify the lengths of dates and permnos arrays
# print(f'Number of dates: {len(dates)}')  # Should be T * N
# print(f'Number of permnos: {len(permnos)}')  # Should be T * N

# # Create the new MultiIndex
# new_index = pd.MultiIndex.from_arrays([dates, permnos], names=['date', 'permno'])

# # Create the DataFrame
# noise_features_df = pd.DataFrame(c_scaled, index=new_index, columns=[f'noise_{i}' for i in range(Pc)])

# # Ensure the new index matches the original DataFrame index
# noise_features_df = noise_features_df.loc[merged_df_padded.index]

# # Add noise features to the original characteristics
# beta_with_noise = pd.concat([merged_df_padded, noise_features_df], axis=1)

# # Replace NaNs back with -2 to maintain the same structure as the original data
# beta_with_noise.fillna(-2, inplace=True)

# # Print to check the DataFrame
# print(beta_with_noise.head())
# print(beta_with_noise.shape)
# print(type(beta_with_noise))


In [ ]:
# import numpy as np
# import pandas as pd

# # Assuming portfolio_rets_df is already defined and loaded

# # Define the list of characteristic codes (same as before)
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Parameters for noise generation
# unique_dates_portfolio = portfolio_rets_df.index.unique()
# T_portfolio = len(unique_dates_portfolio)  # Number of time periods for portfolio returns
# Pc_portfolio = 5  # Number of noise portfolios

# # Calculate the standard deviation of the specified columns
# characteristics_std_portfolio = portfolio_rets_df[characteristics].std().mean()

# # Simulate noise portfolios
# rho_portfolio = np.random.uniform(0.9, 1, Pc_portfolio)
# c_portfolio = np.zeros((T_portfolio, Pc_portfolio))

# # Generate the noise portfolios
# for t in range(1, T_portfolio):
#     c_portfolio[t, :] = rho_portfolio * c_portfolio[t - 1, :] + np.random.normal(0, 1, Pc_portfolio)

# # Normalize noise portfolios
# c_normalized_portfolio = 2 / (T_portfolio + 1) * (c_portfolio.argsort(axis=0).argsort(axis=0) - 1) - 1

# # Scale noise portfolios
# c_scaled_portfolio = c_normalized_portfolio * characteristics_std_portfolio

# # Check the shape of c_scaled_portfolio
# print(c_scaled_portfolio.shape)  # (T_portfolio, Pc_portfolio)

# # Create a DataFrame for noise portfolios with the same index as portfolio_rets_df
# noise_portfolios_df = pd.DataFrame(c_scaled_portfolio, index=portfolio_rets_df.index, columns=[f'noise_{i}' for i in range(Pc_portfolio)])

# # Add noise portfolios to the original portfolio returns
# portfolios_with_noise = pd.concat([portfolio_rets_df, noise_portfolios_df], axis=1)

# # Print to check the DataFrame
# print(portfolios_with_noise.head())
# print(portfolios_with_noise.shape)
# print(type(portfolios_with_noise))



In [ ]:

# # Define the list of characteristic codes
# annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
#                 "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
#                 "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
#                 "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
#                 "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
#                 "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
#                 "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
# quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
#                    "stdacc", "stdcf"]
# monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
#                  "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
#                  "zerotrade", "equal_weighted_portfolio"]
# # Combine all codes
# all_codes = annual_codes + quarterly_codes + monthly_codes
# characteristics = [f'characteristic_{code}' for code in all_codes]

# # Select the required columns including the noise characteristics
# columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)] + ["ret_excess"]
# characteristics_data = beta_with_noise[columns_to_select]



# # # Select the required columns including the noise characteristics
# # characteristics = list(merged_df_cleaned.columns)
# # columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)] + ["ret_excess"]
# # characteristics_data = beta_with_noise[columns_to_select]

# # Function to calculate x_t for a given month using Ridge Regression
# def calculate_xt(df, date, alpha=0):
#     # Filter DataFrame for the given date
#     df_month = df.loc[date]

#     # Separate returns and characteristics
#     r_t = df_month['ret_excess'].values
#     characteristic_columns = df_month.drop(columns=['ret_excess']).columns
#     Z_t_minus_1 = df_month.drop(columns=['ret_excess']).values

#     # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
#     Z_t_minus_1_T = Z_t_minus_1.T
#     Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
#     Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

#     Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

#     # Calculate x_t
#     x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

#     return x_t, characteristic_columns

# # Calculate x_t for all unique dates in the dataset
# def calculate_xt_all_dates(df, alpha=1e-6):
#     xt_results = {}
#     characteristic_columns = None

#     unique_dates = df.index.get_level_values('date').unique()
#     for date in unique_dates:
#         x_t, characteristic_columns = calculate_xt(df, date, alpha)
#         xt_results[date] = x_t

#     # Convert dictionary to DataFrame
#     xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

#     # Assign original characteristic names as column names
#     if characteristic_columns is not None:
#         xt_df.columns = characteristic_columns

#     # Preserve the index name
#     xt_df.index.name = 'date'

#     return xt_df

# # Example usage
# xt_all_dates = calculate_xt_all_dates(characteristics_data)


**MODEL PARAMETER SPECIFICATIONS**

In [ ]:
# # efine the save path
# load_path = '/content/drive/My Drive/Tidy Finance ML dataset'

# # Save the updated data to parquet files
# beta_with_noise.to_parquet(f'{load_path}/final_df.parquet', index=True)
# portfolios_with_noise.to_parquet(f'{load_path}/xt_all_dates.parquet', index=True)

# print("DataFrames have been saved successfully as parquet files.")

In [ ]:
import pandas as pd

# Define the load path
load_path = '/content/drive/My Drive/Tidy Finance ML dataset'

# Load the parquet files
beta_with_noise = pd.read_parquet(f'{load_path}/final_df.parquet')
portfolios_with_noise = pd.read_parquet(f'{load_path}/xt_all_dates.parquet')

# Print the loaded dataframes to verify
print("Loaded final_df:")
print(beta_with_noise.info())
print(beta_with_noise.head())

print("Loaded xt_all_dates:")
print(portfolios_with_noise.info())
print(portfolios_with_noise.head())
portfolios_with_noise = portfolios_with_noise.drop(columns=['characteristic_equal_weighted_portfolio'])

In [ ]:
# import pandas as pd

# # Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# # Define the path to load the files
# load_path = '/content/drive/My Drive/Tidy Finance ML dataset/'

# # Load final_df from the HDF5 file
# beta_with_noise = pd.read_hdf(f'{load_path}/final_df.h5', key='final_df')

# # Load xt_all_dates from the HDF5 file
# portfolios_with_noise = pd.read_hdf(f'{load_path}/xt_all_dates.h5', key='xt_all_dates')

# # Verify the loaded data
# print(beta_with_noise.info())
# print(beta_with_noise.head())
# portfolios_with_noise = portfolios_with_noise.drop(columns=['characteristic_equal_weighted_portfolio'])

# print(portfolios_with_noise.info())
# print(portfolios_with_noise.head())


In [ ]:
Pc=5
# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Select the required columns including the noise characteristics
columns_to_select = characteristics + [f'noise_{i}' for i in range(Pc)]

narratives = columns_to_select
print(narratives)


n_factors = 6
n_narratives = len(columns_to_select)
n_tickers = len(beta_with_noise.index.unique('permno'))

print(n_narratives)
print(n_tickers)



In [ ]:

# Define the columns to select for final_df
columns_to_select_beta = narratives + ["ret_excess", "fwd_ret_excess"]

# Convert specified columns to float16 directly in final_df
for column in columns_to_select_beta:
    beta_with_noise[column] = beta_with_noise[column].astype('float16')

print("Updated final_df:")
print(beta_with_noise.info())
print(beta_with_noise.head())

# Define the columns to select for xt_all_dates
columns_to_select_portfolios = narratives

# # Convert specified columns to float16 directly in xt_all_dates
for column in columns_to_select_portfolios:
    portfolios_with_noise[column] = portfolios_with_noise[column].astype('float16')

print("Updated xt_all_dates:")
print(portfolios_with_noise.info())
print(portfolios_with_noise.head())


**Converting data to float16 for efficiency and saving files**

In [ ]:
# import pandas as pd


# # Define the columns to select
# columns_to_select = narratives + ["ret_excess", "fwd_ret_excess"]

# # Convert specified columns to float16 directly in final_df
# for column in columns_to_select:
#     beta_with_noise[column] = beta_with_noise[column].astype('float16')

# print(beta_with_noise)


# # Define the columns to select
# columns_to_select = narratives

# # Convert specified columns to float16 directly in xt_all_dates
# for column in columns_to_select:
#     portfolios_with_noise[column] = portfolios_with_noise[column].astype('float16')

# print(portfolios_with_noise)



**Saving the figures**

In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr
import time
from itertools import product
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
# Define a custom loss function
def custom_loss(mask_value=-2):
    def loss(y_true, y_pred):
        mask = tf.not_equal(y_true, mask_value)
        mask = tf.cast(mask, dtype=tf.float32)
        mse = tf.square(y_pred - y_true)
        mse *= mask  # Apply the mask to the MSE
        return tf.reduce_sum(mse) / tf.reduce_sum(mask)  # Normalize by the number of valid values
    return loss

# Function to create the model
def make_model(n_tickers, n_narratives, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False, mask_value=-2):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    # Apply Masking layer
    x = Masking(mask_value=mask_value, name='masking_layer')(input_beta)

    # Add hidden layers conditionally
    if num_hidden_layers >= 1:
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)
    if num_hidden_layers >= 2:
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)
    if num_hidden_layers >= 3:
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Combine beta and factor through a dot product
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the model with a custom loss function
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss=custom_loss(mask_value=mask_value), optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both models
    else:
        return model  # Return only the main model

# Function to get train, validation, and test data combined
def get_train_valid_data_combined(merged_data, portfolio_data, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio):
    # Process merged_df_padded data
    train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

    X1_train = train_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)
    X1_val = val_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)
    X1_test = test_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)

    y_train = train_merged.fwd_ret_excess.unstack('permno')
    y_val = val_merged.fwd_ret_excess.unstack('permno')
    y_test = test_merged.fwd_ret_excess.unstack('permno')

    # Process loaded_portfolio_rets data
    X2_train = portfolio_data.loc[train_dates_portfolio]
    X2_val = portfolio_data.loc[val_dates_portfolio]
    X2_test = portfolio_data.loc[test_dates_portfolio]

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test

# Function to compute variable importance

def compute_permutation_variable_importance(model, X1, X2, y):
    # Get the baseline performance
    preds = model.predict([X1, X2])
    baseline_r2 = r2_score(y.values.flatten(), preds.flatten())

    importances = []

    # Iterate over each characteristic
    for i in range(X1.shape[2]):
        X1_copy = X1.copy()
        np.random.shuffle(X1_copy[:, :, i])  # Shuffle the i-th characteristic

        preds_permuted = model.predict([X1_copy, X2])
        permuted_r2 = r2_score(y.values.flatten(), preds_permuted.flatten())

        importance = baseline_r2 - permuted_r2
        importances.append(importance)

    return importances
# Reinitialize strategy for distributed training


def compute_permutation_variable_importance_factor(factor_model, X2):
    # Convert DataFrame to numpy array if necessary
    if isinstance(X2, pd.DataFrame):
        X2 = X2.values

    # Get the baseline predictions
    preds = factor_model.predict(X2)

    # Compute the baseline variance
    baseline_variance = np.var(preds, axis=0).sum()

    importances = []
    for i in range(X2.shape[1]):
        X2_copy = X2.copy()
        np.random.shuffle(X2_copy[:, i])  # Shuffle the i-th feature
        preds_permuted = factor_model.predict(X2_copy)
        permuted_variance = np.var(preds_permuted, axis=0).sum()
        importance = baseline_variance - permuted_variance
        importances.append(importance)

    return importances

def compute_permutation_variable_importance_beta(main_model, X1, X2, y):
    preds = main_model.predict([X1, X2])
    baseline_r2 = r2_score(y.values.flatten(), preds.flatten())

    importances = []
    for i in range(X1.shape[2]):
        X1_copy = X1.copy()
        np.random.shuffle(X1_copy[:, :, i])
        preds_permuted = main_model.predict([X1_copy, X2])
        permuted_r2 = r2_score(y.values.flatten(), preds_permuted.flatten())
        importance = baseline_r2 - permuted_r2
        importances.append(importance)
    return importances



# Seed and strategy setup
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Create a function to convert your data into tf.data.Dataset
def make_dataset(X1, X2, y, batch_size, is_training=True):
    dataset = tf.data.Dataset.from_tensor_slices(((X1, X2), y))
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(X1), seed=SEED)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    return dataset



def calculate_xt(beta_with_noise, result, columns_to_select):
    """
    Calculate x_t for each date using the given characteristics and returns data.

    Parameters:
    - beta_with_noise: DataFrame containing stock characteristics with MultiIndex (date, permno).
    - result: DataFrame containing predicted and true returns with MultiIndex (date, permno).
    - columns_to_select: List of column names representing stock characteristics.

    Returns:
    - xt_df_true: DataFrame containing x_t for each date using true returns.
    - xt_df_pred: DataFrame containing x_t for each date using predicted returns.
    """
    xt_true_list = []
    xt_pred_list = []

    # Ensure index is sorted by date for processing
    beta_with_noise = beta_with_noise.sort_index(level=0)
    result = result.sort_index(level=0)

    # Filter beta_with_noise to only include dates present in result
    common_dates = result.index.get_level_values(0).unique()
    beta_with_noise = beta_with_noise.loc[common_dates]

    for date in common_dates:
        try:
            print(f"Processing date: {date}")

            # Extract Z_{t-1} for the given date
            Z_t_1 = beta_with_noise.loc[date, columns_to_select].replace(-2, np.nan).dropna()

            # Extract true and predicted returns for the given date
            r_t_true = result.loc[date, 'y_true'].dropna()
            r_t_pred = result.loc[date, 'y_pred'].dropna()

            # Ensure Z_t_1 and r_t are not empty and have matching dimensions
            if not Z_t_1.empty and not r_t_true.empty and len(Z_t_1) == len(r_t_true):
                # Convert to numpy arrays with float32
                Z_t_1 = Z_t_1.astype(np.float32).values
                r_t_true = r_t_true.astype(np.float32).values
                r_t_pred = r_t_pred.astype(np.float32).values

                # Compute (Z'_{t-1} Z_{t-1})^{-1}
                Z_t_1_T = Z_t_1.T
                Z_t_1_Z_t_1_inv = np.linalg.inv(Z_t_1_T @ Z_t_1)

                # Compute x_t using true returns
                x_t_true = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_true
                xt_true_list.append(pd.DataFrame(x_t_true.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_true.shape[0])]))

                # Compute x_t using predicted returns
                x_t_pred = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_pred
                xt_pred_list.append(pd.DataFrame(x_t_pred.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_pred.shape[0])]))

            else:
                print(f"Skipping date {date} due to mismatched dimensions or empty data.")

        except Exception as e:
            print(f"Error processing date {date}: {e}")
            continue

    # Concatenate results into DataFrames
    xt_df_true = pd.concat(xt_true_list)
    xt_df_pred = pd.concat(xt_pred_list)

    return xt_df_true, xt_df_pred





early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['num_hidden_layers', 'n_factors', 'l1_reg', 'fold', 'r2_score', 'r2_score_xt']

max_epochs = 100
factor_opts = [5]
l1_reg_opts = [1e-9, 1e-7, 1e-5, 1e-3]
#
hidden_layer_opts = [0,1,2,3]
#0, 1, 2,
param_grid = list(product(hidden_layer_opts, factor_opts))
units = 32
batch_size = 240

YEAR = 12
cv_merged = MultipleTimeSeriesCV(n_splits=20, train_period_length=20* YEAR, val_period_length=2 * YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=20, train_period_length=20* YEAR, val_period_length=2 *YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

start = time.time()

# Create the HDF5 file initially in 'table' format
with pd.HDFStore(str(results_path / 'scores.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
    store.close()

# Define strategy for distributed training
strategy = tf.distribute.MirroredStrategy()

# Initialize DataFrame to store the tuning results
tuning_results = pd.DataFrame(columns=cols)

# Directory to save the model weights
weights_dir = results_path / 'model_weights'
weights_dir.mkdir(exist_ok=True)

# First, iterate over parameter grid to find the best l1_reg for each num_hidden_layers and n_factors combination
for num_hidden_layers, n_factors in tqdm(param_grid):
    for l1_reg in tqdm(l1_reg_opts):
        fold_r2_scores = []
        fold_results = []  # Temporary list to store results for this iteration
        with strategy.scope():
             main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=l1_reg, mask_value=-2)

        for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):

            X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

            # Create datasets
            train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
            val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()

            # Distribute the datasets
            train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
            val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

            # Calculate the number of steps per epoch
            steps_per_epoch = max(len(X1_train) // batch_size, 1)
            validation_steps = max(len(X1_val) // batch_size, 1)

            # Training the whole model
            main_model.fit(train_dist_dataset,
                           validation_data=val_dist_dataset,
                           epochs=max_epochs,
                           verbose=0,
                           steps_per_epoch=steps_per_epoch,
                           validation_steps=validation_steps,
                           callbacks=[early_stopping])

            # Directly call the predict method on the model
            val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
            result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

            # Calculate R²
            r2 = r2_score(result.y_true, result.y_pred)
            fold_r2_scores.append(r2)
            print(f'Model: CA {num_hidden_layers}|No. Factors : {n_factors}|lasso: {l1_reg}|FOLD :{fold}|VAL R^2: {r2}')

            # Append metrics to the list
            fold_results.append({
                'num_hidden_layers': num_hidden_layers,
                'n_factors': n_factors,
                'l1_reg': l1_reg,
                'fold': fold,
                'r2_score': r2
            })

        # Concatenate fold results to the main DataFrame
        tuning_results = pd.concat([tuning_results, pd.DataFrame(fold_results)], ignore_index=True)

        # Save model weights
        weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{l1_reg}.h5'
        main_model.save_weights(weights_path)

# Ensure all columns have consistent data types before saving
tuning_results['num_hidden_layers'] = tuning_results['num_hidden_layers'].astype(int)
tuning_results['n_factors'] = tuning_results['n_factors'].astype(int)
tuning_results['fold'] = tuning_results['fold'].astype(int)
tuning_results['r2_score'] = tuning_results['r2_score'].astype(float)

# Save the tuning results DataFrame
tuning_results.to_hdf(str(results_path / 'tuning_results.h5'), key='tuning_results', format='table')

# Load the tuning results DataFrame
tuning_results = pd.read_hdf(str(results_path / 'tuning_results.h5'), key='tuning_results')

# Calculate the average R² score for each combination of num_hidden_layers, n_factors, and l1_reg
avg_r2_scores = tuning_results.groupby(['num_hidden_layers', 'n_factors', 'l1_reg'])['r2_score'].mean().reset_index()

# Find the best l1_reg for each combination of num_hidden_layers and n_factors based on the highest average R² score
best_l1_reg_per_layer_factor = avg_r2_scores.loc[avg_r2_scores.groupby(['num_hidden_layers', 'n_factors'])['r2_score'].idxmax()]


# Reinitialize strategy for distributed training
strategy = tf.distribute.MirroredStrategy()


import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import r2_score

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []
fold_importances_factor_list = []
fold_importances_beta_list = []
importance_dict = {
    'main_model': {},
    'factor_network': {},
    'beta_network': {}
}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        # fold_scores_val = []
        # fold_val_predictions_list = []
        fold_scores_test = []
        fold_test_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Assuming beta_with_noise and result DataFrames are already defined and available
        xt_df_true_val, xt_df_pred_val = calculate_xt(beta_with_noise, result, columns_to_select)


        # Ensure alignment and matching shapes
        xt_df_true_val = xt_df_true_val.align(xt_df_pred_val, join='inner')[0]
        xt_df_pred_val = xt_df_pred_val.align(xt_df_true_val, join='inner')[0]

        # Flatten the DataFrames
        y_true_flat_val = xt_df_true_val.values.flatten()
        y_pred_flat_val = xt_df_pred_val.values.flatten()

        # Calculate R²
        r2_xt_val = r2_score(y_true_flat_val, y_pred_flat_val)

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}|VAL XT : {r2_xt_val}')

        # fold_scores_val.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2, r2_xt_val])

        # # Save validation predictions for this fold
        # fold_val_predictions_list.append(result)

        # # Save validation predictions for this fold
        # fold_val_predictions_df = pd.concat(fold_val_predictions_list)

        # print(f'VAL PREDICTION : {fold_val_predictions_df.head()}')

#        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        # fold_scores_df_val = pd.DataFrame(fold_scores_val, columns=cols)
        # print(f'VAL SCORES : {fold_scores_df_val.head()}')
        # fold_scores_df_val.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()


        # Calculate R²
        r2_test = r2_score(test_result.y_true, test_result.y_pred)

        # Assuming beta_with_noise and result DataFrames are already defined and available
        xt_df_true_test, xt_df_pred_test = calculate_xt(beta_with_noise, test_result, columns_to_select)

        # Ensure alignment and matching shapes
        xt_df_true_test = xt_df_true_test.align(xt_df_pred_test, join='inner')[0]
        xt_df_pred_test = xt_df_pred_test.align(xt_df_true_test, join='inner')[0]

        # Flatten the DataFrames
        y_true_flat_test = xt_df_true_test.values.flatten()
        y_pred_flat_test = xt_df_pred_test.values.flatten()

        # Calculate R²
        r2_xt_test = r2_score(y_true_flat_test, y_pred_flat_test)

        # Save test predictions for this fold
        fold_test_predictions_list.append(test_result)

        # Save validation predictions for this fold

        # Save test predictions to HDF5
        fold_test_predictions_df = pd.concat(fold_test_predictions_list)
        fold_test_predictions_df.to_hdf(str(results_path / 'test_predictions.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')

        # Add metadata to test predictions
        test_result['num_hidden_layers'] = num_hidden_layers
        test_result['n_factors'] = n_factors
        test_result['best_l1_reg'] = best_l1_reg
        test_result['fold'] = fold

                # Save test predictions to HDF5
        # Save test predictions to HDF5
        # Remove metadata columns before saving to maintain consistency with existing table
        test_result[['y_pred', 'y_true']].to_hdf(str(results_path / 'test_predictions.h5'),
                                                key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}',
                                                append=True, format='table')



        # fold_test_predictions_list.append(result)

        # # Save validation predictions for this fold
        # fold_test_predictions_df = pd.concat(fold_test_predictions_list)

        # print(f'TEST PREDICTION : {fold_test_predictions_df.head()}')
        # fold_test_predictions_df.to_hdf(str(results_path / 'test_predictions.h5'), key=f'fold_{fold}', append=True, format='table')


        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_list.append(importances)

        # Compute and save importances for the factor network
        importances_factor = compute_permutation_variable_importance_factor(factor_network_model, X2_test)
        fold_importances_factor_list.append(importances_factor)

        # Compute and save importances for the beta network
        importances_beta = compute_permutation_variable_importance_beta(main_model, X1_test, X2_test, y_test)
        fold_importances_beta_list.append(importances_beta)

        # Save the importances for heatmap creation
        if num_hidden_layers not in importance_dict['main_model']:
            importance_dict['main_model'][num_hidden_layers] = {}
        if n_factors not in importance_dict['main_model'][num_hidden_layers]:
            importance_dict['main_model'][num_hidden_layers][n_factors] = []
        importance_dict['main_model'][num_hidden_layers][n_factors].append(importances)

        if num_hidden_layers not in importance_dict['factor_network']:
            importance_dict['factor_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['factor_network'][num_hidden_layers]:
            importance_dict['factor_network'][num_hidden_layers][n_factors] = []
        importance_dict['factor_network'][num_hidden_layers][n_factors].append(importances_factor)

        if num_hidden_layers not in importance_dict['beta_network']:
            importance_dict['beta_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['beta_network'][num_hidden_layers]:
            importance_dict['beta_network'][num_hidden_layers][n_factors] = []
        importance_dict['beta_network'][num_hidden_layers][n_factors].append(importances_beta)

        # Save and plot importances for the main model for each combination
        avg_importances = np.mean(fold_importances_list, axis=0)
        importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
        importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
        importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
        importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_df.to_hdf(str(results_path / f'variable_importance_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the main model for each combination
        top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Main Model with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the factor network for each combination
        avg_importances_factor = np.mean(fold_importances_factor_list, axis=0)
        importances_factor_df = pd.DataFrame(avg_importances_factor, index=columns_to_select, columns=['importance'])
        importances_factor_df['importance_normalized'] = importances_factor_df['importance'] / importances_factor_df['importance'].sum()
        importances_factor_df = importances_factor_df.reset_index().rename(columns={'index': 'variable'})
        importances_factor_df['variable'] = importances_factor_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_factor_df.to_hdf(str(results_path / f'variable_importance_factor_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_factor = importances_factor_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_factor['importance_normalized'], y=top_20_importances_factor['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Factor Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the beta network for each combination
        avg_importances_beta = np.mean(fold_importances_beta_list, axis=0)
        importances_beta_df = pd.DataFrame(avg_importances_beta, index=columns_to_select, columns=['importance'])
        importances_beta_df['importance_normalized'] = importances_beta_df['importance'] / importances_beta_df['importance'].sum()
        importances_beta_df = importances_beta_df.reset_index().rename(columns={'index': 'variable'})
        importances_beta_df['variable'] = importances_beta_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_beta_df.to_hdf(str(results_path / f'variable_importance_beta_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_beta = importances_beta_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_beta['importance_normalized'], y=top_20_importances_beta['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Beta Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

# Main Model Heatmap Data
importance_df_main = create_heatmap_data(importance_dict['main_model'], columns_to_select)
heatmap_df_main = importance_df_main.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_main.index = heatmap_df_main.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_main.columns = heatmap_df_main.columns.astype(int)


# Factor Network Heatmap Data
importance_df_factor = create_heatmap_data(importance_dict['factor_network'], columns_to_select)
heatmap_df_factor = importance_df_factor.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df_factor.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_factor.columns = heatmap_df_factor.columns.astype(int)


# Beta Network Heatmap Data
importance_df_beta = create_heatmap_data(importance_dict['beta_network'], columns_to_select)
heatmap_df_beta = importance_df_beta.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_beta.index = heatmap_df_beta.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_beta.columns = heatmap_df_beta.columns.astype(int)



# Save heatmap data
heatmap_df_main.to_hdf(str(results_path / 'heatmap_data_main.h5'), key='heatmap_main', format='table')
heatmap_df_factor.to_hdf(str(results_path / 'heatmap_data_factor.h5'), key='heatmap_factor', format='table')
heatmap_df_beta.to_hdf(str(results_path / 'heatmap_data_beta.h5'), key='heatmap_beta', format='table')


# Plot the heatmaps
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_main, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance with noise for Autoencoder model')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_factor, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Characteristics Importance - Factor Network')
# plt.show()


plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for factor network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_beta, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances Heatmap - Beta Network')
# plt.show()

plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_beta, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for beta network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()







**Only predictions**

In [ ]:
# Define a custom loss function
def custom_loss(mask_value=-2):
    def loss(y_true, y_pred):
        mask = tf.not_equal(y_true, mask_value)
        mask = tf.cast(mask, dtype=tf.float32)
        mse = tf.square(y_pred - y_true)
        mse *= mask  # Apply the mask to the MSE
        return tf.reduce_sum(mse) / tf.reduce_sum(mask)  # Normalize by the number of valid values
    return loss

# Function to create the model
def make_model(n_tickers, n_narratives, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False, mask_value=-2):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    # Apply Masking layer
    x = Masking(mask_value=mask_value, name='masking_layer')(input_beta)

    # Add hidden layers conditionally
    if num_hidden_layers >= 1:
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)
    if num_hidden_layers >= 2:
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)
    if num_hidden_layers >= 3:
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Combine beta and factor through a dot product
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the model with a custom loss function
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss=custom_loss(mask_value=mask_value), optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both models
    else:
        return model  # Return only the main model

# Function to get train, validation, and test data combined
def get_train_valid_data_combined(merged_data, portfolio_data, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio):
    # Process merged_df_padded data
    train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

    X1_train = train_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)
    X1_val = val_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)
    X1_test = test_merged.loc[:, columns_to_select].values.reshape(-1, n_tickers, n_narratives)

    y_train = train_merged.fwd_ret_excess.unstack('permno')
    y_val = val_merged.fwd_ret_excess.unstack('permno')
    y_test = test_merged.fwd_ret_excess.unstack('permno')

    # Process loaded_portfolio_rets data
    X2_train = portfolio_data.loc[train_dates_portfolio]
    X2_val = portfolio_data.loc[val_dates_portfolio]
    X2_test = portfolio_data.loc[test_dates_portfolio]

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test

# Function to compute variable importance

def compute_permutation_variable_importance(model, X1, X2, y):
    # Get the baseline performance
    preds = model.predict([X1, X2])
    baseline_r2 = r2_score(y.values.flatten(), preds.flatten())

    importances = []

    # Iterate over each characteristic
    for i in range(X1.shape[2]):
        X1_copy = X1.copy()
        np.random.shuffle(X1_copy[:, :, i])  # Shuffle the i-th characteristic

        preds_permuted = model.predict([X1_copy, X2])
        permuted_r2 = r2_score(y.values.flatten(), preds_permuted.flatten())

        importance = baseline_r2 - permuted_r2
        importances.append(importance)

    return importances
# Reinitialize strategy for distributed training


def compute_permutation_variable_importance_factor(factor_model, X2):
    # Convert DataFrame to numpy array if necessary
    if isinstance(X2, pd.DataFrame):
        X2 = X2.values

    # Get the baseline predictions
    preds = factor_model.predict(X2)

    # Compute the baseline variance
    baseline_variance = np.var(preds, axis=0).sum()

    importances = []
    for i in range(X2.shape[1]):
        X2_copy = X2.copy()
        np.random.shuffle(X2_copy[:, i])  # Shuffle the i-th feature
        preds_permuted = factor_model.predict(X2_copy)
        permuted_variance = np.var(preds_permuted, axis=0).sum()
        importance = baseline_variance - permuted_variance
        importances.append(importance)

    return importances

def compute_permutation_variable_importance_beta(main_model, X1, X2, y):
    preds = main_model.predict([X1, X2])
    baseline_r2 = r2_score(y.values.flatten(), preds.flatten())

    importances = []
    for i in range(X1.shape[2]):
        X1_copy = X1.copy()
        np.random.shuffle(X1_copy[:, :, i])
        preds_permuted = main_model.predict([X1_copy, X2])
        permuted_r2 = r2_score(y.values.flatten(), preds_permuted.flatten())
        importance = baseline_r2 - permuted_r2
        importances.append(importance)
    return importances



# Seed and strategy setup
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Create a function to convert your data into tf.data.Dataset
def make_dataset(X1, X2, y, batch_size, is_training=True):
    dataset = tf.data.Dataset.from_tensor_slices(((X1, X2), y))
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(X1), seed=SEED)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    return dataset


import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

def calculate_xt(beta_with_noise, result, columns_to_select):
    """
    Calculate x_t for each date using the given characteristics and returns data.

    Parameters:
    - beta_with_noise: DataFrame containing stock characteristics with MultiIndex (date, permno).
    - result: DataFrame containing predicted and true returns with MultiIndex (date, permno).
    - columns_to_select: List of column names representing stock characteristics.

    Returns:
    - xt_df_true: DataFrame containing x_t for each date using true returns.
    - xt_df_pred: DataFrame containing x_t for each date using predicted returns.
    """
    xt_true_list = []
    xt_pred_list = []

    # Ensure index is sorted by date for processing
    beta_with_noise = beta_with_noise.sort_index(level=0)
    result = result.sort_index(level=0)

    # Filter beta_with_noise to only include dates present in result
    common_dates = result.index.get_level_values(0).unique()
    beta_with_noise = beta_with_noise.loc[common_dates - pd.DateOffset(months=1)]

    for date in common_dates:
        try:
            print(f"Processing date: {date}")

            # Extract Z_{t-1} for the given date
            Z_t_1 = beta_with_noise.loc[date - pd.DateOffset(months=1), columns_to_select].replace(-2, np.nan).dropna()

            # Extract true and predicted returns for the given date
            r_t_true = result.loc[date, 'y_true'].dropna()
            r_t_pred = result.loc[date, 'y_pred'].dropna()

            # Align Z_t_1 with r_t based on permno
            common_permnos = Z_t_1.index.intersection(r_t_true.index)
            Z_t_1 = Z_t_1.loc[common_permnos]
            r_t_true = r_t_true.loc[common_permnos]
            r_t_pred = r_t_pred.loc[common_permnos]

            # Ensure Z_t_1 and r_t are not empty and have matching dimensions
            if not Z_t_1.empty and not r_t_true.empty and len(Z_t_1) == len(r_t_true):
                # Convert to numpy arrays with float32
                Z_t_1 = Z_t_1.astype(np.float32).values
                r_t_true = r_t_true.astype(np.float32).values
                r_t_pred = r_t_pred.astype(np.float32).values

                # Compute (Z'_{t-1} Z_{t-1})^{-1}
                Z_t_1_T = Z_t_1.T
                Z_t_1_Z_t_1_inv = np.linalg.inv(Z_t_1_T @ Z_t_1)

                # Compute x_t using true returns
                x_t_true = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_true
                xt_true_list.append(pd.DataFrame(x_t_true.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_true.shape[0])]))

                # Compute x_t using predicted returns
                x_t_pred = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_pred
                xt_pred_list.append(pd.DataFrame(x_t_pred.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_pred.shape[0])]))

            else:
                print(f"Skipping date {date} due to mismatched dimensions or empty data.")
                print(f"Z_t_1 shape: {Z_t_1.shape if Z_t_1 is not None else 'None'}, r_t_true shape: {r_t_true.shape if r_t_true is not None else 'None'}, r_t_pred shape: {r_t_pred.shape if r_t_pred is not None else 'None'}")

        except Exception as e:
            print(f"Error processing date {date}: {e}")
            continue

    # Concatenate results into DataFrames
    if xt_true_list:
        xt_df_true = pd.concat(xt_true_list)
    else:
        xt_df_true = pd.DataFrame()

    if xt_pred_list:
        xt_df_pred = pd.concat(xt_pred_list)
    else:
        xt_df_pred = pd.DataFrame()

    return xt_df_true, xt_df_pred



early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['num_hidden_layers', 'n_factors', 'l1_reg', 'fold', 'r2_score', 'r2_score_xt']

max_epochs = 100
factor_opts = [5]
l1_reg_opts = [1e-9, 1e-7, 1e-5, 1e-3]
#
hidden_layer_opts = [0,1,2,3]
#0, 1, 2,
param_grid = list(product(hidden_layer_opts, factor_opts))
units = 32
batch_size = 240

YEAR = 12
cv_merged = MultipleTimeSeriesCV(n_splits=20, train_period_length=20* YEAR, val_period_length=2 * YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=20, train_period_length=20* YEAR, val_period_length=2 *YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

start = time.time()

# Create the HDF5 file initially in 'table' format
with pd.HDFStore(str(results_path / 'scores.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
    store.close()






# Directory to save the model weights
weights_dir = results_path / 'model_weights'
weights_dir.mkdir(exist_ok=True)


# Load the tuning results DataFrame
tuning_results = pd.read_hdf(str(results_path / 'tuning_results.h5'), key='tuning_results')

# Calculate the average R² score for each combination of num_hidden_layers, n_factors, and l1_reg
avg_r2_scores = tuning_results.groupby(['num_hidden_layers', 'n_factors', 'l1_reg'])['r2_score'].mean().reset_index()

# Find the best l1_reg for each combination of num_hidden_layers and n_factors based on the highest average R² score
best_l1_reg_per_layer_factor = avg_r2_scores.loc[avg_r2_scores.groupby(['num_hidden_layers', 'n_factors'])['r2_score'].idxmax()]


# Reinitialize strategy for distributed training
strategy = tf.distribute.MirroredStrategy()



import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import r2_score

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []
fold_importances_factor_list = []
fold_importances_beta_list = []
importance_dict = {
    'main_model': {},
    'factor_network': {},
    'beta_network': {}
}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):


        fold_scores = []
        fold_test_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)
        test_dist_dataset = strategy.experimental_distribute_dataset(test_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Assuming beta_with_noise and result DataFrames are already defined and available
        xt_df_true_val, xt_df_pred_val = calculate_xt(beta_with_noise, result, columns_to_select)

        # Ensure alignment and matching shapes
        xt_df_true_val, xt_df_pred_val = xt_df_true_val.align(xt_df_pred_val, join='inner')

        # Flatten the DataFrames
        y_true_flat_val = xt_df_true_val.values.flatten()
        y_pred_flat_val = xt_df_pred_val.values.flatten()

        # Calculate R²
        r2_xt_val = r2_score(y_true_flat_val, y_pred_flat_val)
        r2_val = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2_val}|VAL XT : {r2_xt_val}')

        # Save validation predictions for this fold
        # fold_val_predictions_list.append(result)

        # # Save validation predictions to HDF5
        # fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        # fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')

        # # Save validation scores for this fold
        # fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2_val, r2_xt_val])

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dist_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        # Calculate R²
        r2_test = r2_score(test_result.y_true, test_result.y_pred)

        # Assuming beta_with_noise and test_result DataFrames are already defined and available
        xt_df_true_test, xt_df_pred_test = calculate_xt(beta_with_noise, test_result, columns_to_select)

        # Ensure alignment and matching shapes
        xt_df_true_test, xt_df_pred_test = xt_df_true_test.align(xt_df_pred_test, join='inner')

        # Flatten the DataFrames
        y_true_flat_test = xt_df_true_test.values.flatten()
        y_pred_flat_test = xt_df_pred_test.values.flatten()

        # Calculate R²
        r2_xt_test = r2_score(y_true_flat_test, y_pred_flat_test)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|TEST R^2: {r2_test}|TEST XT : {r2_xt_test}')

        # Save test predictions for this fold
        fold_test_predictions_list.append(test_result)

        # Save test predictions to HDF5
        fold_test_predictions_df = pd.concat(fold_test_predictions_list)
        fold_test_predictions_df.to_hdf(str(results_path / 'test_predictions.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')

        # Add metadata to test predictions
        test_result['num_hidden_layers'] = num_hidden_layers
        test_result['n_factors'] = n_factors
        test_result['best_l1_reg'] = best_l1_reg
        test_result['fold'] = fold

                # Save test predictions to HDF5
        # Save test predictions to HDF5
        # Remove metadata columns before saving to maintain consistency with existing table
        test_result[['y_pred', 'y_true']].to_hdf(str(results_path / 'test_predictions.h5'),
                                                key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}',
                                                append=True, format='table')
        # Save fold scores
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2_test, r2_xt_test])
        fold_scores_df = pd.DataFrame(fold_scores, columns=['num_hidden_layers', 'n_factors', 'best_l1_reg', 'fold', 'r2', 'r2_xt_test'])

        # Save fold scores to HDF5
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')



        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_list.append(importances)

        # Compute and save importances for the factor network
        importances_factor = compute_permutation_variable_importance_factor(factor_network_model, X2_test)
        fold_importances_factor_list.append(importances_factor)

        # Compute and save importances for the beta network
        importances_beta = compute_permutation_variable_importance_beta(main_model, X1_test, X2_test, y_test)
        fold_importances_beta_list.append(importances_beta)

        # Save the importances for heatmap creation
        if num_hidden_layers not in importance_dict['main_model']:
            importance_dict['main_model'][num_hidden_layers] = {}
        if n_factors not in importance_dict['main_model'][num_hidden_layers]:
            importance_dict['main_model'][num_hidden_layers][n_factors] = []
        importance_dict['main_model'][num_hidden_layers][n_factors].append(importances)

        if num_hidden_layers not in importance_dict['factor_network']:
            importance_dict['factor_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['factor_network'][num_hidden_layers]:
            importance_dict['factor_network'][num_hidden_layers][n_factors] = []
        importance_dict['factor_network'][num_hidden_layers][n_factors].append(importances_factor)

        if num_hidden_layers not in importance_dict['beta_network']:
            importance_dict['beta_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['beta_network'][num_hidden_layers]:
            importance_dict['beta_network'][num_hidden_layers][n_factors] = []
        importance_dict['beta_network'][num_hidden_layers][n_factors].append(importances_beta)

        # Save and plot importances for the main model for each combination
        avg_importances = np.mean(fold_importances_list, axis=0)
        importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
        importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
        importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
        importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_df.to_hdf(str(results_path / f'variable_importance_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the main model for each combination
        top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Main Model with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the factor network for each combination
        avg_importances_factor = np.mean(fold_importances_factor_list, axis=0)
        importances_factor_df = pd.DataFrame(avg_importances_factor, index=columns_to_select, columns=['importance'])
        importances_factor_df['importance_normalized'] = importances_factor_df['importance'] / importances_factor_df['importance'].sum()
        importances_factor_df = importances_factor_df.reset_index().rename(columns={'index': 'variable'})
        importances_factor_df['variable'] = importances_factor_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_factor_df.to_hdf(str(results_path / f'variable_importance_factor_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_factor = importances_factor_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_factor['importance_normalized'], y=top_20_importances_factor['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Factor Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the beta network for each combination
        avg_importances_beta = np.mean(fold_importances_beta_list, axis=0)
        importances_beta_df = pd.DataFrame(avg_importances_beta, index=columns_to_select, columns=['importance'])
        importances_beta_df['importance_normalized'] = importances_beta_df['importance'] / importances_beta_df['importance'].sum()
        importances_beta_df = importances_beta_df.reset_index().rename(columns={'index': 'variable'})
        importances_beta_df['variable'] = importances_beta_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_beta_df.to_hdf(str(results_path / f'variable_importance_beta_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_beta = importances_beta_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_beta['importance_normalized'], y=top_20_importances_beta['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Beta Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

# Main Model Heatmap Data
importance_df_main = create_heatmap_data(importance_dict['main_model'], columns_to_select)
heatmap_df_main = importance_df_main.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_main.index = heatmap_df_main.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_main.columns = heatmap_df_main.columns.astype(int)


# Factor Network Heatmap Data
importance_df_factor = create_heatmap_data(importance_dict['factor_network'], columns_to_select)
heatmap_df_factor = importance_df_factor.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df_factor.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_factor.columns = heatmap_df_factor.columns.astype(int)


# Beta Network Heatmap Data
importance_df_beta = create_heatmap_data(importance_dict['beta_network'], columns_to_select)
heatmap_df_beta = importance_df_beta.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_beta.index = heatmap_df_beta.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_beta.columns = heatmap_df_beta.columns.astype(int)



# Save heatmap data
heatmap_df_main.to_hdf(str(results_path / 'heatmap_data_main.h5'), key='heatmap_main', format='table')
heatmap_df_factor.to_hdf(str(results_path / 'heatmap_data_factor.h5'), key='heatmap_factor', format='table')
heatmap_df_beta.to_hdf(str(results_path / 'heatmap_data_beta.h5'), key='heatmap_beta', format='table')


# Plot the heatmaps
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_main, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance with noise for Autoencoder model')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_factor, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Characteristics Importance - Factor Network')
# plt.show()


plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for factor network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_beta, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances Heatmap - Beta Network')
# plt.show()

plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_beta, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for beta network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()







In [ ]:
print(X1_test.shape,X2_test.shape, y_test.shape)

In [ ]:
print(importances_factor_df)

In [ ]:

# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            avg_importances = np.maximum(avg_importances, 0)  # Ensure no negative importance
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])


# Main Model Heatmap Data
importance_df_main = create_heatmap_data(importance_dict['main_model'], columns_to_select)
heatmap_df_main = importance_df_main.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_main.index = heatmap_df_main.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_main.columns = heatmap_df_main.columns.astype(int)


# Factor Network Heatmap Data
importance_df_factor = create_heatmap_data(importance_dict['factor_network'], columns_to_select)
heatmap_df_factor = importance_df_factor.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df_factor.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_factor.columns = heatmap_df_factor.columns.astype(int)


# Beta Network Heatmap Data
importance_df_beta = create_heatmap_data(importance_dict['beta_network'], columns_to_select)
heatmap_df_beta = importance_df_beta.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_beta.index = heatmap_df_beta.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_beta.columns = heatmap_df_beta.columns.astype(int)



# Save heatmap data
heatmap_df_main.to_hdf(str(results_path / 'heatmap_data_main_removed_negative_numbers.h5'), key='heatmap_main', format='table')
heatmap_df_factor.to_hdf(str(results_path / 'heatmap_data_facto_removed_negative_numbers.h5'), key='heatmap_factor', format='table')
heatmap_df_beta.to_hdf(str(results_path / 'heatmap_data_beta_removed_negative_numbers.h5'), key='heatmap_beta', format='table')


# Plot the heatmaps
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_main, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance with noise for Autoencoder model')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_main_model_{int(num_hidden_layers)}_{int(n_factors)}_removed_negative_numbers.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_factor, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Characteristics Importance - Factor Network')
# plt.show()


plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for factor network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_factor_network_{int(num_hidden_layers)}_{int(n_factors)}_removed_negative_numbers.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_beta, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances Heatmap - Beta Network')
# plt.show()

plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_beta, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for beta network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_beta_network_{int(num_hidden_layers)}_{int(n_factors)}_removed_negative_numbers.png'))
plt.show()







In [ ]:
# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])


print(importance_df_main)
print(importance_df_factor)
print(importance_df_beta)

In [ ]:
# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            avg_importances = np.maximum(avg_importances, 0)  # Ensure no negative importance
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

print(importance_df_main)
print(importance_df_factor)
print(importance_df_beta)

**only plots**

In [ ]:

# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

# Main Model Heatmap Data
importance_df_main = create_heatmap_data(importance_dict['main_model'], columns_to_select)
heatmap_df_main = importance_df_main.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_main.index = heatmap_df_main.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_main.columns = heatmap_df_main.columns.astype(int)


# Factor Network Heatmap Data
importance_df_factor = create_heatmap_data(importance_dict['factor_network'], columns_to_select)
heatmap_df_factor = importance_df_factor.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df_factor.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_factor.columns = heatmap_df_factor.columns.astype(int)


# Beta Network Heatmap Data
importance_df_beta = create_heatmap_data(importance_dict['beta_network'], columns_to_select)
heatmap_df_beta = importance_df_beta.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_beta.index = heatmap_df_beta.index.str.replace('characteristic_', '')
# Convert the column names to integers
heatmap_df_beta.columns = heatmap_df_beta.columns.astype(int)



# Save heatmap data
heatmap_df_main.to_hdf(str(results_path / 'heatmap_data_main.h5'), key='heatmap_main', format='table')
heatmap_df_factor.to_hdf(str(results_path / 'heatmap_data_factor.h5'), key='heatmap_factor', format='table')
heatmap_df_beta.to_hdf(str(results_path / 'heatmap_data_beta.h5'), key='heatmap_beta', format='table')


# Plot the heatmaps
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_main, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance with noise for Autoencoder model')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_factor, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Characteristics Importance - Factor Network')
# plt.show()


plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for factor network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_beta, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances Heatmap - Beta Network')
# plt.show()

plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_beta, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for beta network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()







In [ ]:
def calculate_xt(beta_with_noise, result, columns_to_select):
    """
    Calculate x_t for each date using the given characteristics and returns data.

    Parameters:
    - beta_with_noise: DataFrame containing stock characteristics with MultiIndex (date, permno).
    - result: DataFrame containing predicted and true returns with MultiIndex (date, permno).
    - columns_to_select: List of column names representing stock characteristics.

    Returns:
    - xt_df_true: DataFrame containing x_t for each date using true returns.
    - xt_df_pred: DataFrame containing x_t for each date using predicted returns.
    """
    xt_true_list = []
    xt_pred_list = []

    # Ensure index is sorted by date for processing
    beta_with_noise = beta_with_noise.sort_index(level=0)
    result = result.sort_index(level=0)

    # Filter beta_with_noise to only include dates present in result
    common_dates = result.index.get_level_values(0).unique()
    beta_with_noise = beta_with_noise.loc[common_dates]

    for date in common_dates:
        try:
            print(f"Processing date: {date}")

            # Extract Z_{t-1} for the given date
            Z_t_1 = beta_with_noise.loc[date, columns_to_select].replace(-2, np.nan).dropna()

            # Extract true and predicted returns for the given date
            r_t_true = result.loc[date, 'y_true'].dropna()
            r_t_pred = result.loc[date, 'y_pred'].dropna()

            # Ensure Z_t_1 and r_t are not empty and have matching dimensions
            if not Z_t_1.empty and not r_t_true.empty and len(Z_t_1) == len(r_t_true):
                # Convert to numpy arrays with float32
                Z_t_1 = Z_t_1.astype(np.float32).values
                r_t_true = r_t_true.astype(np.float32).values
                r_t_pred = r_t_pred.astype(np.float32).values

                # Compute (Z'_{t-1} Z_{t-1})^{-1}
                Z_t_1_T = Z_t_1.T
                Z_t_1_Z_t_1_inv = np.linalg.inv(Z_t_1_T @ Z_t_1)

                # Compute x_t using true returns
                x_t_true = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_true
                xt_true_list.append(pd.DataFrame(x_t_true.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_true.shape[0])]))

                # Compute x_t using predicted returns
                x_t_pred = Z_t_1_Z_t_1_inv @ Z_t_1_T @ r_t_pred
                xt_pred_list.append(pd.DataFrame(x_t_pred.reshape(1, -1), index=[date], columns=[f'x_{i+1}' for i in range(x_t_pred.shape[0])]))

            else:
                print(f"Skipping date {date} due to mismatched dimensions or empty data.")

        except Exception as e:
            print(f"Error processing date {date}: {e}")
            continue

    # Concatenate results into DataFrames
    xt_df_true = pd.concat(xt_true_list)
    xt_df_pred = pd.concat(xt_pred_list)

    return xt_df_true, xt_df_pred

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import r2_score

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []
fold_importances_factor_list = []
fold_importances_beta_list = []
importance_dict = {
    'main_model': {},
    'factor_network': {},
    'beta_network': {}
}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        fold_scores = []
        fold_val_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Assuming beta_with_noise and result DataFrames are already defined and available
        xt_df_true_val, xt_df_pred_val = calculate_xt(beta_with_noise, result, columns_to_select)


        # Ensure alignment and matching shapes
        xt_df_true_val = xt_df_true_val.align(xt_df_pred_val, join='inner')[0]
        xt_df_pred_val = xt_df_pred_val.align(xt_df_true_val, join='inner')[0]

        # Flatten the DataFrames
        y_true_flat_val = xt_df_true_val.values.flatten()
        y_pred_flat_val = xt_df_pred_val.values.flatten()

        # Calculate R²
        r2_xt_val = r2_score(y_true_flat_val, y_pred_flat_val)

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}|VAL XT : {r2_xt_val}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2])

        # Save validation predictions for this fold
        fold_val_predictions_list.append(result)

        # Save validation predictions for this fold
        fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        fold_scores_df = pd.DataFrame(fold_scores, columns=cols)
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()


        # Calculate R²
        r2_test = r2_score(test_result.y_true, test_result.y_pred)

        # Assuming beta_with_noise and result DataFrames are already defined and available
        xt_df_true_test, xt_df_pred_test = calculate_xt(beta_with_noise, test_result, columns_to_select)

        # Ensure alignment and matching shapes
        xt_df_true_test = xt_df_true_test.align(xt_df_pred_test, join='inner')[0]
        xt_df_pred_test = xt_df_pred_test.align(xt_df_true_test, join='inner')[0]

        # Flatten the DataFrames
        y_true_flat_test = xt_df_true_test.values.flatten()
        y_pred_flat_test = xt_df_pred_test.values.flatten()

        # Calculate R²
        r2_xt_test = r2_score(y_true_flat_test, y_pred_flat_test)

        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|TEST R^2: {r2_test}|TEST XT : {r2_xt_test}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2, r2_xt_test])

        # Save test predictions for this fold
        test_predictions_list.append(test_result)

        # Add metadata to test predictions
        test_result['num_hidden_layers'] = num_hidden_layers
        test_result['n_factors'] = n_factors
        test_result['best_l1_reg'] = best_l1_reg
        test_result['fold'] = fold

        # Save test predictions to HDF5
        test_result.to_hdf(str(results_path / 'test_predictions.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')

        # Convert fold scores to DataFrame and add metadata
        fold_scores_df = pd.DataFrame(fold_scores, columns=['num_hidden_layers', 'n_factors', 'best_l1_reg', 'fold', 'r2', 'r2_xt_test'])
        fold_scores_df['num_hidden_layers'] = num_hidden_layers
        fold_scores_df['n_factors'] = n_factors
        fold_scores_df['best_l1_reg'] = best_l1_reg
        fold_scores_df['fold'] = fold

        # Save fold scores to HDF5
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')



        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_list.append(importances)

        # Compute and save importances for the factor network
        importances_factor = compute_permutation_variable_importance_factor(factor_network_model, X2_test)
        fold_importances_factor_list.append(importances_factor)

        # Compute and save importances for the beta network
        importances_beta = compute_permutation_variable_importance_beta(main_model, X1_test, X2_test, y_test)
        fold_importances_beta_list.append(importances_beta)

        # Save the importances for heatmap creation
        if num_hidden_layers not in importance_dict['main_model']:
            importance_dict['main_model'][num_hidden_layers] = {}
        if n_factors not in importance_dict['main_model'][num_hidden_layers]:
            importance_dict['main_model'][num_hidden_layers][n_factors] = []
        importance_dict['main_model'][num_hidden_layers][n_factors].append(importances)

        if num_hidden_layers not in importance_dict['factor_network']:
            importance_dict['factor_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['factor_network'][num_hidden_layers]:
            importance_dict['factor_network'][num_hidden_layers][n_factors] = []
        importance_dict['factor_network'][num_hidden_layers][n_factors].append(importances_factor)

        if num_hidden_layers not in importance_dict['beta_network']:
            importance_dict['beta_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['beta_network'][num_hidden_layers]:
            importance_dict['beta_network'][num_hidden_layers][n_factors] = []
        importance_dict['beta_network'][num_hidden_layers][n_factors].append(importances_beta)

        # Save and plot importances for the main model for each combination
        avg_importances = np.mean(fold_importances_list, axis=0)
        importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
        importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
        importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
        importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_df.to_hdf(str(results_path / f'variable_importance_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the main model for each combination
        top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Main Model with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the factor network for each combination
        avg_importances_factor = np.mean(fold_importances_factor_list, axis=0)
        importances_factor_df = pd.DataFrame(avg_importances_factor, index=columns_to_select, columns=['importance'])
        importances_factor_df['importance_normalized'] = importances_factor_df['importance'] / importances_factor_df['importance'].sum()
        importances_factor_df = importances_factor_df.reset_index().rename(columns={'index': 'variable'})
        importances_factor_df['variable'] = importances_factor_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_factor_df.to_hdf(str(results_path / f'variable_importance_factor_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_factor = importances_factor_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_factor['importance_normalized'], y=top_20_importances_factor['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Factor Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the beta network for each combination
        avg_importances_beta = np.mean(fold_importances_beta_list, axis=0)
        importances_beta_df = pd.DataFrame(avg_importances_beta, index=columns_to_select, columns=['importance'])
        importances_beta_df['importance_normalized'] = importances_beta_df['importance'] / importances_beta_df['importance'].sum()
        importances_beta_df = importances_beta_df.reset_index().rename(columns={'index': 'variable'})
        importances_beta_df['variable'] = importances_beta_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_beta_df.to_hdf(str(results_path / f'variable_importance_beta_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_beta = importances_beta_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_beta['importance_normalized'], y=top_20_importances_beta['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Beta Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

# Main Model Heatmap Data
importance_df_main = create_heatmap_data(importance_dict['main_model'], columns_to_select)
heatmap_df_main = importance_df_main.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_main.index = heatmap_df_main.index.str.replace('characteristic_', '')


# Factor Network Heatmap Data
importance_df_factor = create_heatmap_data(importance_dict['factor_network'], columns_to_select)
heatmap_df_factor = importance_df_factor.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df_factor.index.str.replace('characteristic_', '')


# Beta Network Heatmap Data
importance_df_beta = create_heatmap_data(importance_dict['beta_network'], columns_to_select)
heatmap_df_beta = importance_df_beta.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_beta.index = heatmap_df_beta.index.str.replace('characteristic_', '')

# Save heatmap data
heatmap_df_main.to_hdf(str(results_path / 'heatmap_data_main.h5'), key='heatmap_main', format='table')
heatmap_df_factor.to_hdf(str(results_path / 'heatmap_data_factor.h5'), key='heatmap_factor', format='table')
heatmap_df_beta.to_hdf(str(results_path / 'heatmap_data_beta.h5'), key='heatmap_beta', format='table')


# Plot the heatmaps
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_main, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance with noise for Autoencoder model')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_factor, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Characteristics Importance - Factor Network')
# plt.show()


plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for factor network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_beta, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances Heatmap - Beta Network')
# plt.show()

plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_beta, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for beta network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()








# Display the calculated x_t DataFrames
print("True Returns x_t:")
print(xt_df_true.head())

print("Predicted Returns x_t:")
print(xt_df_pred.head())

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import r2_score

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []
fold_importances_factor_list = []
fold_importances_beta_list = []
importance_dict = {
    'main_model': {},
    'factor_network': {},
    'beta_network': {}
}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        fold_scores = []
        fold_val_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Assuming beta_with_noise and result DataFrames are already defined and available
        xt_df_true_val, xt_df_pred_val = calculate_xt(beta_with_noise, result, columns_to_select)


        # Ensure alignment and matching shapes
        xt_df_true_val = xt_df_true_val.align(xt_df_pred_val, join='inner')[0]
        xt_df_pred_val = xt_df_pred_val.align(xt_df_true_val, join='inner')[0]

        # Flatten the DataFrames
        y_true_flat_val = xt_df_true_val.values.flatten()
        y_pred_flat_val = xt_df_pred_val.values.flatten()

        # Calculate R²
        r2_xt_val = r2_score(y_true_flat_val, y_pred_flat_val)

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}|VAL XT : {r2_xt_val}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2])

        # Save validation predictions for this fold
        fold_val_predictions_list.append(result)

        # Save validation predictions for this fold
        fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        fold_scores_df = pd.DataFrame(fold_scores, columns=cols)
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()


        # Calculate R²
        r2_test = r2_score(test_result.y_true, test_result.y_pred)

        # Assuming beta_with_noise and result DataFrames are already defined and available
        xt_df_true_test, xt_df_pred_test = calculate_xt(beta_with_noise, test_result, columns_to_select)

        # Ensure alignment and matching shapes
        xt_df_true_test = xt_df_true_test.align(xt_df_pred_test, join='inner')[0]
        xt_df_pred_test = xt_df_pred_test.align(xt_df_true_test, join='inner')[0]

        # Flatten the DataFrames
        y_true_flat_test = xt_df_true_test.values.flatten()
        y_pred_flat_test = xt_df_pred_test.values.flatten()

        # Calculate R²
        r2_xt_test = r2_score(y_true_flat_test, y_pred_flat_test)

        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|TEST R^2: {r2_test}|TEST XT : {r2_xt_test}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2, r2_xt_test])

        # Save test predictions for this fold
        test_predictions_list.append(test_result)

        # Add metadata to test predictions
        test_result['num_hidden_layers'] = num_hidden_layers
        test_result['n_factors'] = n_factors
        test_result['best_l1_reg'] = best_l1_reg
        test_result['fold'] = fold

        # Save test predictions to HDF5
        test_result.to_hdf(str(results_path / 'test_predictions.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')

        # Convert fold scores to DataFrame and add metadata
        fold_scores_df = pd.DataFrame(fold_scores, columns=['num_hidden_layers', 'n_factors', 'best_l1_reg', 'fold', 'r2', 'r2_xt_test'])
        fold_scores_df['num_hidden_layers'] = num_hidden_layers
        fold_scores_df['n_factors'] = n_factors
        fold_scores_df['best_l1_reg'] = best_l1_reg
        fold_scores_df['fold'] = fold

        # Save fold scores to HDF5
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'model_{num_hidden_layers}_{n_factors}_{best_l1_reg}_fold_{fold}', append=True, format='table')



        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_list.append(importances)

        # Compute and save importances for the factor network
        importances_factor = compute_permutation_variable_importance_factor(factor_network_model, X2_test)
        fold_importances_factor_list.append(importances_factor)

        # Compute and save importances for the beta network
        importances_beta = compute_permutation_variable_importance_beta(main_model, X1_test, X2_test, y_test)
        fold_importances_beta_list.append(importances_beta)

        # Save the importances for heatmap creation
        if num_hidden_layers not in importance_dict['main_model']:
            importance_dict['main_model'][num_hidden_layers] = {}
        if n_factors not in importance_dict['main_model'][num_hidden_layers]:
            importance_dict['main_model'][num_hidden_layers][n_factors] = []
        importance_dict['main_model'][num_hidden_layers][n_factors].append(importances)

        if num_hidden_layers not in importance_dict['factor_network']:
            importance_dict['factor_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['factor_network'][num_hidden_layers]:
            importance_dict['factor_network'][num_hidden_layers][n_factors] = []
        importance_dict['factor_network'][num_hidden_layers][n_factors].append(importances_factor)

        if num_hidden_layers not in importance_dict['beta_network']:
            importance_dict['beta_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['beta_network'][num_hidden_layers]:
            importance_dict['beta_network'][num_hidden_layers][n_factors] = []
        importance_dict['beta_network'][num_hidden_layers][n_factors].append(importances_beta)

        # Save and plot importances for the main model for each combination
        avg_importances = np.mean(fold_importances_list, axis=0)
        importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
        importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
        importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
        importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_df.to_hdf(str(results_path / f'variable_importance_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the main model for each combination
        top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Main Model with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the factor network for each combination
        avg_importances_factor = np.mean(fold_importances_factor_list, axis=0)
        importances_factor_df = pd.DataFrame(avg_importances_factor, index=columns_to_select, columns=['importance'])
        importances_factor_df['importance_normalized'] = importances_factor_df['importance'] / importances_factor_df['importance'].sum()
        importances_factor_df = importances_factor_df.reset_index().rename(columns={'index': 'variable'})
        importances_factor_df['variable'] = importances_factor_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_factor_df.to_hdf(str(results_path / f'variable_importance_factor_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_factor = importances_factor_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_factor['importance_normalized'], y=top_20_importances_factor['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Factor Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

        # Plot the top 20 variable importances for the beta network for each combination
        avg_importances_beta = np.mean(fold_importances_beta_list, axis=0)
        importances_beta_df = pd.DataFrame(avg_importances_beta, index=columns_to_select, columns=['importance'])
        importances_beta_df['importance_normalized'] = importances_beta_df['importance'] / importances_beta_df['importance'].sum()
        importances_beta_df = importances_beta_df.reset_index().rename(columns={'index': 'variable'})
        importances_beta_df['variable'] = importances_beta_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_beta_df.to_hdf(str(results_path / f'variable_importance_beta_{int(num_hidden_layers)}_{int(n_factors)}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_beta = importances_beta_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_beta['importance_normalized'], y=top_20_importances_beta['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Characteristics Importances - Beta Network with noise - $CA_{int(num_hidden_layers)}$, Factors: {int(n_factors)}')
        plt.savefig(str(results_path / f'top_20_importances_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
        plt.show()

# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

# Main Model Heatmap Data
importance_df_main = create_heatmap_data(importance_dict['main_model'], columns_to_select)
heatmap_df_main = importance_df_main.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_main.index = heatmap_df_main.index.str.replace('characteristic_', '')


# Factor Network Heatmap Data
importance_df_factor = create_heatmap_data(importance_dict['factor_network'], columns_to_select)
heatmap_df_factor = importance_df_factor.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df_factor.index.str.replace('characteristic_', '')


# Beta Network Heatmap Data
importance_df_beta = create_heatmap_data(importance_dict['beta_network'], columns_to_select)
heatmap_df_beta = importance_df_beta.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_beta.index = heatmap_df_beta.index.str.replace('characteristic_', '')

# Save heatmap data
heatmap_df_main.to_hdf(str(results_path / 'heatmap_data_main.h5'), key='heatmap_main', format='table')
heatmap_df_factor.to_hdf(str(results_path / 'heatmap_data_factor.h5'), key='heatmap_factor', format='table')
heatmap_df_beta.to_hdf(str(results_path / 'heatmap_data_beta.h5'), key='heatmap_beta', format='table')


# Plot the heatmaps
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_main, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance with noise for Autoencoder model')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_main_model_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_factor, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Characteristics Importance - Factor Network')
# plt.show()


plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for factor network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_beta, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances Heatmap - Beta Network')
# plt.show()

plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_beta, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for beta network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_beta_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()








# Display the calculated x_t DataFrames
print("True Returns x_t:")
print(xt_df_true.head())

print("Predicted Returns x_t:")
print(xt_df_pred.head())

In [ ]:
print(xt_df_true_test)

In [ ]:
print(result.shape)
print(result.index)
print(beta_with_noise.shape)
print(beta_with_noise.index)

In [ ]:
# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_dict = {0: [], 1: [], 2: [], 3: []}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        fold_scores = []
        fold_val_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2])

        # Save validation predictions for this fold
        fold_val_predictions_list.append(result)

        # Save validation predictions for this fold
        fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        fold_scores_df = pd.DataFrame(fold_scores, columns=cols)
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        # Save test predictions for this fold
        test_predictions_list.append(test_result)

        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_dict[num_hidden_layers].append(importances)

# Save all test predictions
all_test_predictions_df = pd.concat(test_predictions_list)
all_test_predictions_df.to_hdf(str(results_path / 'test_predictions.h5'), key='all_folds', append=True, format='table')

# Calculate average variable importances across all folds for each num_hidden_layers
avg_importances_dict = {hidden_layers: np.mean(importances, axis=0) for hidden_layers, importances in fold_importances_dict.items()}

# Create a DataFrame for the heatmap
heatmap_df = pd.DataFrame(avg_importances_dict, index=columns_to_select)

# Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, annot=False, cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Variables')
plt.title('Variable Importances for Different Numbers of Hidden Layers')
plt.savefig(str(results_path / 'variable_importance_heatmap.png'))
plt.show()


In [ ]:

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []
fold_importances_factor_list = []
fold_importances_beta_list = []
# fold_importances_dict = {0: [], 1: [], 2: [], 3: []}
# fold_importances_dict_factor = {0: [], 1: [], 2: [], 3: []}
# fold_importances_dict_beta = {0: [], 1: [], 2: [], 3: []}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        fold_scores = []
        fold_val_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2])

        # Save validation predictions for this fold
        fold_val_predictions_list.append(result)

        # Save validation predictions for this fold
        fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        fold_scores_df = pd.DataFrame(fold_scores, columns=cols)
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        # Save test predictions for this fold
        test_predictions_list.append(test_result)

        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        # fold_importances_dict[num_hidden_layers].append(importances)

        importances_factor = compute_permutation_variable_importance_factor(factor_network_model, X2_test)
        # fold_importances_dict_factor[num_hidden_layers].append(importances_factor)

        importances_beta = compute_permutation_variable_importance_beta(main_model, X1_test, X2_test, y_test)
        # fold_importances_dict_beta[num_hidden_layers].append(importances_beta)

        fold_importances_list.append(importances)
        fold_importances_factor_list.append(importances_factor)
        fold_importances_beta_list.append(importances_beta)

        # Save and plot importances for the main model for each combination
        avg_importances = np.mean(fold_importances_list, axis=0)
        importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
        importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
        importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
        importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_df.to_hdf(str(results_path / f'variable_importance_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the main model for each combination
        top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Main Model - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_main_model_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

        # Save and plot importances for the factor network for each combination
        avg_importances_factor = np.mean(fold_importances_factor_list, axis=0)
        importances_factor_df = pd.DataFrame(avg_importances_factor, index=columns_to_select, columns=['importance'])
        importances_factor_df['importance_normalized'] = importances_factor_df['importance'] / importances_factor_df['importance'].sum()
        importances_factor_df = importances_factor_df.reset_index().rename(columns={'index': 'variable'})

        # Save the importance DataFrame
        importances_factor_df.to_hdf(str(results_path / f'variable_importance_factor_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the factor network for each combination
        top_20_importances_factor = importances_factor_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_factor['importance_normalized'], y=top_20_importances_factor['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Factor Network - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_factor_network_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

        # Save and plot importances for the beta network for each combination
        avg_importances_beta = np.mean(fold_importances_beta_list, axis=0)
        importances_beta_df = pd.DataFrame(avg_importances_beta, index=columns_to_select, columns=['importance'])
        importances_beta_df['importance_normalized'] = importances_beta_df['importance'] / importances_beta_df['importance'].sum()
        importances_beta_df = importances_beta_df.reset_index().rename(columns={'index': 'variable'})
        importances_beta_df['variable'] = importances_beta_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_beta_df.to_hdf(str(results_path / f'variable_importance_beta_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the beta network for each combination
        top_20_importances_beta = importances_beta_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_beta['importance_normalized'], y=top_20_importances_beta['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Beta Network - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_beta_network_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

# # Calculate average variable importances across all folds for each num_hidden_layers
# avg_importances_dict = {hidden_layers: np.mean(importances, axis=0) for hidden_layers, importances in fold_importances_dict.items()}


# # Calculate average variable importances across all folds for each num_hidden_layers
# avg_importances_dict_factor = {hidden_layers: np.mean(importances_factor, axis=0) for hidden_layers, importances_factor in fold_importances_dict_factor.items()}

# # Calculate average variable importances across all folds for each num_hidden_layers
# avg_importances_dict_beta = {hidden_layers: np.mean(importances_beta, axis=0) for hidden_layers, importances_beta in fold_importances_dict_beta.items()}



# # Create a DataFrame for the heatmap
# heatmap_df = pd.DataFrame(avg_importances_dict, index=columns_to_select)

# # Plot the heatmap
# plt.figure(figsize=(12, 10))
# sns.heatmap(heatmap_df, annot=False, cmap='viridis', cbar_kws={'label': 'Importance'})
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances for Different Numbers of Hidden Layers')
# plt.savefig(str(results_path / 'variable_importance_heatmap.png'))
# plt.show()



# # Create a DataFrame for the heatmap
# heatmap_df_factor = pd.DataFrame(avg_importances_dict_factor, index=columns_to_select)

# # Plot the heatmap
# plt.figure(figsize=(12, 10))
# sns.heatmap(heatmap_df_factor, annot=False, cmap='viridis', cbar_kws={'label': 'Importance'})
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances for Different Numbers of Hidden Layers')
# plt.savefig(str(results_path / 'variable_importance_heatmap.png'))
# plt.show()


# # Create a DataFrame for the heatmap
# heatmap_df = pd.DataFrame(avg_importances_dict, index=columns_to_select)

# # Plot the heatmap
# plt.figure(figsize=(12, 10))
# sns.heatmap(heatmap_df, annot=False, cmap='viridis', cbar_kws={'label': 'Importance'})
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances for Different Numbers of Hidden Layers')
# plt.savefig(str(results_path / 'variable_importance_heatmap.png'))
# plt.show()
# print("Predictions and variable importance computation complete.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import r2_score

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []
fold_importances_factor_list = []
fold_importances_beta_list = []
importance_dict = {}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        fold_scores = []
        fold_val_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2])

        # Save validation predictions for this fold
        fold_val_predictions_list.append(result)

        # Save validation predictions for this fold
        fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        fold_scores_df = pd.DataFrame(fold_scores, columns=cols)
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        # Save test predictions for this fold
        test_predictions_list.append(test_result)

        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_list.append(importances)

        # Save the importances for heatmap creation
        if num_hidden_layers not in importance_dict:
            importance_dict[num_hidden_layers] = {}
        if n_factors not in importance_dict[num_hidden_layers]:
            importance_dict[num_hidden_layers][n_factors] = []
        importance_dict[num_hidden_layers][n_factors].append(importances)

        # Save and plot importances for the main model for each combination
        avg_importances = np.mean(fold_importances_list, axis=0)
        importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
        importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
        importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
        importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_df.to_hdf(str(results_path / f'variable_importance_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the main model for each combination
        top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Main Model - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_main_model_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

        # Compute and save importances for the factor network
        importances_factor = compute_permutation_variable_importance_factor(factor_network_model, X2_test)
        fold_importances_factor_list.append(importances_factor)
        avg_importances_factor = np.mean(fold_importances_factor_list, axis=0)
        importances_factor_df = pd.DataFrame(avg_importances_factor, index=columns_to_select, columns=['importance'])
        importances_factor_df['importance_normalized'] = importances_factor_df['importance'] / importances_factor_df['importance'].sum()
        importances_factor_df = importances_factor_df.reset_index().rename(columns={'index': 'variable'})
        importances_factor_df['variable'] = importances_factor_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_factor_df.to_hdf(str(results_path / f'variable_importance_factor_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the factor network for each combination
        top_20_importances_factor = importances_factor_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_factor['importance_normalized'], y=top_20_importances_factor['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Factor Network - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_factor_network_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

        # Compute and save importances for the beta network
        importances_beta = compute_permutation_variable_importance_beta(main_model, X1_test, X2_test, y_test)
        fold_importances_beta_list.append(importances_beta)
        avg_importances_beta = np.mean(fold_importances_beta_list, axis=0)
        importances_beta_df = pd.DataFrame(avg_importances_beta, index=columns_to_select, columns=['importance'])
        importances_beta_df['importance_normalized'] = importances_beta_df['importance'] / importances_beta_df['importance'].sum()
        importances_beta_df = importances_beta_df.reset_index().rename(columns={'index': 'variable'})
        importances_beta_df['variable'] = importances_beta_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_beta_df.to_hdf(str(results_path / f'variable_importance_beta_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the beta network for each combination
        top_20_importances_beta = importances_beta_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_beta['importance_normalized'], y=top_20_importances_beta['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Beta Network - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_beta_network_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

# Aggregate importances for heatmap
importance_data = []
for num_hidden_layers, factors_dict in importance_dict.items():
    for n_factors, importances_list in factors_dict.items():
        avg_importances = np.mean(importances_list, axis=0)
        for var_idx, importance in enumerate(avg_importances):
            importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])

importance_df = pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

# Pivot the DataFrame for heatmap plotting
heatmap_df = importance_df.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')

# Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, annot=True, fmt='.2f', cmap='viridis')
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Variables')
plt.title('Variable Importances Heatmap')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import r2_score

# Initialize lists to store final test predictions and importances
test_predictions_list = []
fold_importances_list = []
fold_importances_factor_list = []
fold_importances_beta_list = []
importance_dict = {
    'main_model': {},
    'factor_network': {},
    'beta_network': {}
}

start = time.time()

# Final evaluation using the best l1_reg value for each num_hidden_layers and n_factors
for _, row in tqdm(best_l1_reg_per_layer_factor.iterrows()):
    num_hidden_layers = row['num_hidden_layers']
    n_factors = row['n_factors']
    best_l1_reg = row['l1_reg']

    # Reload the best model weights
    weights_path = weights_dir / f'weights_{int(num_hidden_layers)}_{int(n_factors)}_{best_l1_reg}.h5'
    with strategy.scope():
        main_model, factor_network_model = make_model(n_tickers, n_factors=n_factors, n_narratives=n_narratives, num_hidden_layers=num_hidden_layers, return_components=True, l1_reg=best_l1_reg, mask_value=-2)
        main_model.load_weights(weights_path)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(beta_with_noise), cv_portfolio.split(portfolios_with_noise))):
        fold_scores = []
        fold_val_predictions_list = []

        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(beta_with_noise, portfolios_with_noise, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Create datasets
        train_dataset = make_dataset(X1_train, X2_train, y_train, batch_size).repeat()
        val_dataset = make_dataset(X1_val, X2_val, y_val, batch_size, is_training=False).repeat()
        test_dataset = make_dataset(X1_test, X2_test, y_test, batch_size, is_training=False).repeat()

        # Distribute the datasets
        train_dist_dataset = strategy.experimental_distribute_dataset(train_dataset)
        val_dist_dataset = strategy.experimental_distribute_dataset(val_dataset)

        # Calculate the number of steps per epoch
        steps_per_epoch = max(len(X1_train) // batch_size, 1)
        validation_steps = max(len(X1_val) // batch_size, 1)
        test_steps = max(len(X1_test) // batch_size, 1)

        # Evaluate the model on validation set
        val_preds = main_model.predict(val_dist_dataset, steps=validation_steps)
        result = pd.DataFrame({'y_pred': val_preds.flatten(), 'y_true': y_val.stack().values}, index=y_val.stack().index).replace(-2, np.nan).dropna()

        # Calculate R²
        r2 = r2_score(result.y_true, result.y_pred)
        print(f'Time :{time.time()-start:.2f}|num_hidden_layers: {num_hidden_layers}|n_factors: {n_factors}|lasso: {best_l1_reg}|FOLD{fold}|VAL R^2: {r2}')
        fold_scores.append([num_hidden_layers, n_factors, best_l1_reg, fold, r2])

        # Save validation predictions for this fold
        fold_val_predictions_list.append(result)

        # Save validation predictions for this fold
        fold_val_predictions_df = pd.concat(fold_val_predictions_list)
        fold_val_predictions_df.to_hdf(str(results_path / 'validation_predictions.h5'), key=f'fold_{fold}', append=True, format='table')

        fold_scores_df = pd.DataFrame(fold_scores, columns=cols)
        fold_scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'fold_{fold}', append=True, format='table')

        # Evaluate the model on test set
        test_preds = main_model.predict(test_dataset, steps=test_steps)
        test_result = pd.DataFrame({'y_pred': test_preds.flatten(), 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        # Save test predictions for this fold
        test_predictions_list.append(test_result)

        # Compute variable importance using the best model for this fold and test set
        importances = compute_permutation_variable_importance(main_model, X1_test, X2_test, y_test)
        fold_importances_list.append(importances)

        # Compute and save importances for the factor network
        importances_factor = compute_permutation_variable_importance_factor(factor_network_model, X2_test)
        fold_importances_factor_list.append(importances_factor)

        # Compute and save importances for the beta network
        importances_beta = compute_permutation_variable_importance_beta(main_model, X1_test, X2_test, y_test)
        fold_importances_beta_list.append(importances_beta)

        # Save the importances for heatmap creation
        if num_hidden_layers not in importance_dict['main_model']:
            importance_dict['main_model'][num_hidden_layers] = {}
        if n_factors not in importance_dict['main_model'][num_hidden_layers]:
            importance_dict['main_model'][num_hidden_layers][n_factors] = []
        importance_dict['main_model'][num_hidden_layers][n_factors].append(importances)

        if num_hidden_layers not in importance_dict['factor_network']:
            importance_dict['factor_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['factor_network'][num_hidden_layers]:
            importance_dict['factor_network'][num_hidden_layers][n_factors] = []
        importance_dict['factor_network'][num_hidden_layers][n_factors].append(importances_factor)

        if num_hidden_layers not in importance_dict['beta_network']:
            importance_dict['beta_network'][num_hidden_layers] = {}
        if n_factors not in importance_dict['beta_network'][num_hidden_layers]:
            importance_dict['beta_network'][num_hidden_layers][n_factors] = []
        importance_dict['beta_network'][num_hidden_layers][n_factors].append(importances_beta)

        # Save and plot importances for the main model for each combination
        avg_importances = np.mean(fold_importances_list, axis=0)
        importances_df = pd.DataFrame(avg_importances, index=columns_to_select, columns=['importance'])
        importances_df['importance_normalized'] = importances_df['importance'] / importances_df['importance'].sum()
        importances_df = importances_df.reset_index().rename(columns={'index': 'variable'})
        importances_df['variable'] = importances_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_df.to_hdf(str(results_path / f'variable_importance_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        # Plot the top 20 variable importances for the main model for each combination
        top_20_importances = importances_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances['importance_normalized'], y=top_20_importances['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Main Model - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_main_model_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

        # Plot the top 20 variable importances for the factor network for each combination
        avg_importances_factor = np.mean(fold_importances_factor_list, axis=0)
        importances_factor_df = pd.DataFrame(avg_importances_factor, index=columns_to_select, columns=['importance'])
        importances_factor_df['importance_normalized'] = importances_factor_df['importance'] / importances_factor_df['importance'].sum()
        importances_factor_df = importances_factor_df.reset_index().rename(columns={'index': 'variable'})
        importances_factor_df['variable'] = importances_factor_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_factor_df.to_hdf(str(results_path / f'variable_importance_factor_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_factor = importances_factor_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_factor['importance_normalized'], y=top_20_importances_factor['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Factor Network - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_factor_network_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

        # Plot the top 20 variable importances for the beta network for each combination
        avg_importances_beta = np.mean(fold_importances_beta_list, axis=0)
        importances_beta_df = pd.DataFrame(avg_importances_beta, index=columns_to_select, columns=['importance'])
        importances_beta_df['importance_normalized'] = importances_beta_df['importance'] / importances_beta_df['importance'].sum()
        importances_beta_df = importances_beta_df.reset_index().rename(columns={'index': 'variable'})
        importances_beta_df['variable'] = importances_beta_df['variable'].str.replace('characteristic_', '')

        # Save the importance DataFrame
        importances_beta_df.to_hdf(str(results_path / f'variable_importance_beta_{num_hidden_layers}_{n_factors}.h5'), key='all_folds', append=True, format='table')

        top_20_importances_beta = importances_beta_df.sort_values('importance_normalized', ascending=False).head(20)
        plt.figure(figsize=(12, 10))
        sns.barplot(x=top_20_importances_beta['importance_normalized'], y=top_20_importances_beta['variable'])
        plt.xlabel('Normalized Importance')
        plt.ylabel('Variables')
        plt.title(f'Top 20 Normalized Variable Importances - Beta Network - Layers: {num_hidden_layers}, Factors: {n_factors}')
        plt.savefig(str(results_path / f'top_20_importances_beta_network_{num_hidden_layers}_{n_factors}.png'))
        plt.show()

# Aggregate importances for heatmap
def create_heatmap_data(network_importance_dict, columns_to_select):
    importance_data = []
    for num_hidden_layers, factors_dict in network_importance_dict.items():
        for n_factors, importances_list in factors_dict.items():
            avg_importances = np.mean(importances_list, axis=0)
            for var_idx, importance in enumerate(avg_importances):
                importance_data.append([num_hidden_layers, n_factors, columns_to_select[var_idx], importance])
    return pd.DataFrame(importance_data, columns=['num_hidden_layers', 'n_factors', 'variable', 'importance'])

# Main Model Heatmap Data
importance_df_main = create_heatmap_data(importance_dict['main_model'], columns_to_select)
heatmap_df_main = importance_df_main.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_main.index = heatmap_df.index.str.replace('characteristic_', '')


# Factor Network Heatmap Data
importance_df_factor = create_heatmap_data(importance_dict['factor_network'], columns_to_select)
heatmap_df_factor = importance_df_factor.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df.index.str.replace('characteristic_', '')


# Beta Network Heatmap Data
importance_df_beta = create_heatmap_data(importance_dict['beta_network'], columns_to_select)
heatmap_df_beta = importance_df_beta.pivot_table(values='importance', index='variable', columns='num_hidden_layers', aggfunc='mean')
# Remove the prefix "characteristic_" from the index names
heatmap_df_beta.index = heatmap_df.index.str.replace('characteristic_', '')

# Save heatmap data
heatmap_df_main.to_hdf(str(results_path / 'heatmap_data_main.h5'), key='heatmap_main', format='table')
heatmap_df_factor.to_hdf(str(results_path / 'heatmap_data_factor.h5'), key='heatmap_factor', format='table')
heatmap_df_beta.to_hdf(str(results_path / 'heatmap_data_beta.h5'), key='heatmap_beta', format='table')


# Plot the heatmaps
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_main, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance with noise for Autoencoder model')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_main_model_{num_hidden_layers}_{n_factors}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_factor, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Characteristics Importance - Factor Network')
# plt.show()


plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for factor network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_factor_network_{int(num_hidden_layers)}_{int(n_factors)}.png'))
plt.show()

# plt.figure(figsize=(15, 20))
# sns.heatmap(heatmap_df_beta, annot=True, fmt='.2f', cmap='viridis')
# plt.xlabel('Number of Hidden Layers')
# plt.ylabel('Variables')
# plt.title('Variable Importances Heatmap - Beta Network')
# plt.show()

plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_beta, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Characteristics')
plt.title('Characteristics Importance for beta network with noise ')
# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')
plt.savefig(str(results_path / f'characteristics_importance_beta_network_{num_hidden_layers}_{n_factors}.png'))
plt.show()






In [ ]:

# Remove the prefix "characteristic_" from the index names
heatmap_df_factor.index = heatmap_df.index.str.replace('characteristic_', '')

# Plot the heatmap
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df_factor, annot=False, fmt='.2f', cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Variables')
plt.title('Variable Importances for Different Numbers of Hidden Layers')

# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')

# plt.savefig('/mnt/data/variable_importance_heatmap.png')
plt.show()


In [ ]:
print(importances_df)

In [ ]:
print(len(columns_to_select))
print(columns_to_select)

In [ ]:


      # Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, annot=False, cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Variables')
plt.title('Variable Importances for Different Numbers of Hidden Layers')
plt.savefig(str(results_path / 'variable_importance_heatmap.png'))
plt.show()


In [ ]:
print(heatmap_df)

In [ ]:
# Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, annot=False, cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Variables')
plt.title('Variable Importances for Different Numbers of Hidden Layers')
plt.savefig('variable_importance_heatmap.png')
plt.show()

In [ ]:

# Remove the prefix "characteristic_" from the index names
heatmap_df.index = heatmap_df.index.str.replace('characteristic_', '')

# Plot the heatmap
plt.figure(figsize=(15, 20))
ax = sns.heatmap(heatmap_df, annot=False, cmap='viridis', cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Variables')
plt.title('Variable Importances for Different Numbers of Hidden Layers')

# Color the "noise_" rows in red
for i, label in enumerate(ax.get_yticklabels()):
    if 'noise_' in label.get_text():
        label.set_color('red')

# plt.savefig('/mnt/data/variable_importance_heatmap.png')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import ListedColormap

# Remove the prefix "characteristic_" from the index names
heatmap_df.index = heatmap_df.index.str.replace('characteristic_', '')

# Create a mask for 'noise_' rows
mask = heatmap_df.index.str.startswith('noise_')

# Define a custom colormap
cmap = sns.color_palette("viridis", as_cmap=True)
custom_colors = ["red" if row else cmap(i / (len(heatmap_df.index) - 1)) for i, row in enumerate(mask)]
custom_cmap = ListedColormap(custom_colors)

# Plot the heatmap with the custom colormap
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, annot=False, cmap=custom_cmap, cbar_kws={'label': 'Importance'})
plt.xlabel('Number of Hidden Layers')
plt.ylabel('Variables')
plt.title('Variable Importances for Different Numbers of Hidden Layers')

#plt.savefig('/mnt/data/variable_importance_heatmap.png')
plt.show()